# 📊 CRM Dashboard — Importador Meta → Google Sheets

Este notebook importa automaticamente os dados exportados do **WhatsApp Manager (Meta)** para a sua planilha do Google Sheets, que alimenta o dashboard.

---

## ▶️ Como usar (passo a passo)

1. **Execute a Célula 1** — instala as dependências (só precisa fazer uma vez por sessão)
2. **Execute a Célula 2** — faz o upload dos seus arquivos ZIP
3. **Execute a Célula 3** — processa e envia os dados para o Google Sheets

> 💡 **Dica:** Para executar uma célula, clique nela e pressione o botão **▶** que aparece à esquerda, ou use `Shift + Enter`.

---

### 📁 Como preparar os arquivos

1. Exporte os CSVs do WhatsApp Manager (Meta)
2. **Renomeie cada CSV** com o nome do modelo de mensagem (ex: `TRACKER.csv`, `ONIX_PLUS.csv`)
3. Compacte os CSVs de cada marca em um ZIP com o nome da marca (ex: `GMSP.zip`)
4. Faça o upload na Célula 2

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 1 — Instalação das dependências
# Execute esta célula primeiro (apenas uma vez por sessão)
# ═══════════════════════════════════════════════════════════

import subprocess
subprocess.run(['pip', 'install', 'requests', 'pandas', '--quiet'], check=True)

print('✅ Dependências instaladas com sucesso!')
print('👉 Agora execute a Célula 2 para fazer o upload dos seus arquivos ZIP.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 2 — Upload dos arquivos ZIP
# O nome do ZIP deve ser o nome da marca (ex: GMSP.zip)
# Os CSVs dentro do ZIP devem ter o nome do modelo (ex: TRACKER.csv)
# ═══════════════════════════════════════════════════════════

from google.colab import files
import os

print('📂 Selecione os arquivos ZIP das marcas...')
print('   (Você pode selecionar múltiplos arquivos de uma vez)')
print()

uploaded = files.upload()

if uploaded:
    print(f'\n✅ {len(uploaded)} arquivo(s) carregado(s):')
    for nome in uploaded.keys():
        tamanho = len(uploaded[nome]) / 1024
        print(f'   📦 {nome} ({tamanho:.1f} KB)')
    print()
    print('👉 Agora execute a Célula 3 para importar os dados.')
else:
    print('⚠️  Nenhum arquivo foi carregado. Tente novamente.')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CÉLULA 3 — Processamento e envio para o Google Sheets
# ═══════════════════════════════════════════════════════════

import zipfile, pandas as pd, requests, json, base64, os, io, re
from datetime import datetime

# URL do Apps Script (já configurada)
APPS_SCRIPT_URL = "https://script.google.com/macros/s/AKfycbzjM9Eyelg9_1NedNfqBS9RJ_pTA78KLwujC7DkgzwNyB7dGNykWslZ4ajqqUr3bfIG/exec"

def extrair_nome_modelo(nome_arquivo):
    """Usa o nome do arquivo CSV como nome do modelo.
    Ex: 'TRACKER.csv' -> 'TRACKER'
    Se ainda tem o nome padrao do Meta (insights_...), usa a data de inicio."""
    base = os.path.splitext(os.path.basename(nome_arquivo))[0]
    if not base.lower().startswith('insights_'):
        return base
    match = re.search(r'insights_(\d{4}-\d{2}-\d{2})_to_', base)
    if match:
        return f'Disparo {match.group(1)}'
    base = re.sub(r'insights_\d{4}-\d{2}-\d{2}_to_\d{4}-\d{2}-\d{2}', '', base)
    base = re.sub(r'\s*\(\d+\)\s*$', '', base)
    base = base.strip('_- ')
    return base if base else os.path.splitext(os.path.basename(nome_arquivo))[0]

def extrair_data_csv(nome_arquivo):
    match = re.search(r'(\d{4}-\d{2}-\d{2})_to_', nome_arquivo)
    return match.group(1) if match else datetime.today().strftime('%Y-%m-%d')

def processar_csv(conteudo_bytes, nome_arquivo, marca):
    try:
        df = pd.read_csv(io.BytesIO(conteudo_bytes))
    except Exception:
        try:
            df = pd.read_csv(io.BytesIO(conteudo_bytes), encoding='latin-1')
        except Exception as e:
            print(f'     ⚠️  Erro ao ler {nome_arquivo}: {e}')
            return None

    df.columns = [c.strip() for c in df.columns]
    col_metrica = next((c for c in df.columns if c.upper() == 'METRIC'), None)
    col_count   = next((c for c in df.columns if c.upper() == 'COUNT'), None)
    col_button  = next((c for c in df.columns if 'BUTTON' in c.upper()), None)

    if col_metrica is None or col_count is None:
        print(f'     ⚠️  Colunas METRIC/COUNT nao encontradas em {nome_arquivo}')
        return None

    nome_modelo = extrair_nome_modelo(nome_arquivo)
    enviadas = entregues = cliques = cliques_parar = 0

    for _, row in df.iterrows():
        metrica = str(row.get(col_metrica, '')).strip()
        try:
            count = int(float(str(row.get(col_count, 0)).replace(',', '')))
        except:
            count = 0

        if 'enviadas' in metrica.lower() or metrica.lower() == 'sent':
            enviadas += count
        elif 'entregues' in metrica.lower() or metrica.lower() == 'delivered':
            entregues += count
        elif 'clique' in metrica.lower() or 'click' in metrica.lower() or 'button' in metrica.lower():
            botao = str(row.get(col_button, '')).strip().lower() if col_button else ''
            if 'parar' in botao or 'stop' in botao or 'opt-out' in botao:
                cliques_parar += count
            else:
                cliques += count

    return {
        'nome':          nome_modelo,
        'marca':         marca,
        'tipo':          'WhatsApp',
        'status':        'concluida',
        'enviadas':      enviadas,
        'entregues':     entregues,
        'cliques':       cliques,
        'cliques_parar': cliques_parar,
        'mensagem':      '',
        'data':          extrair_data_csv(nome_arquivo)
    }

def enviar_para_sheets(registros):
    LOTE = 20
    total_inseridos = total_atualizados = 0
    for i in range(0, len(registros), LOTE):
        lote = registros[i:i+LOTE]
        dados_b64 = base64.b64encode(json.dumps(lote, ensure_ascii=False).encode('utf-8')).decode('ascii')
        try:
            resp = requests.get(APPS_SCRIPT_URL, params={'action': 'importar', 'data': dados_b64}, allow_redirects=True, timeout=60)
            resultado = resp.json()
            if resultado.get('sucesso'):
                msg = resultado.get('mensagem', '')
                nums = re.findall(r'\d+', msg)
                if len(nums) >= 2:
                    total_inseridos  += int(nums[0])
                    total_atualizados += int(nums[1])
            else:
                print(f'  ⚠️  Erro no lote {i//LOTE+1}: {resultado.get("erro", "desconhecido")}')
        except Exception as e:
            print(f'  ⚠️  Erro: {e}')
    return total_inseridos, total_atualizados

# ─────────────────────────────────────────────────────────
# EXECUÇÃO PRINCIPAL
# ─────────────────────────────────────────────────────────
print('=' * 60)
print('  IMPORTADOR META → GOOGLE SHEETS CRM DASHBOARD')
print('=' * 60)

zips_disponiveis = [f for f in os.listdir('.') if f.lower().endswith('.zip')]

if not zips_disponiveis:
    print('⚠️  Nenhum arquivo ZIP encontrado!')
    print('   Execute a Célula 2 primeiro para fazer o upload dos arquivos.')
else:
    print(f'  Encontrados {len(zips_disponiveis)} arquivo(s) ZIP:')
    for z in zips_disponiveis:
        print(f'    📦 {z}')
    print()

    todos_registros = []

    for zip_file in zips_disponiveis:
        marca = os.path.splitext(zip_file)[0]
        print(f'  Processando marca: {marca}')
        print('  ' + '-' * 40)
        try:
            with zipfile.ZipFile(zip_file, 'r') as zf:
                csvs = [n for n in zf.namelist() if n.lower().endswith('.csv')]
                print(f'  Encontrados {len(csvs)} arquivo(s) CSV')
                for csv_nome in sorted(csvs):
                    with zf.open(csv_nome) as f:
                        conteudo = f.read()
                    reg = processar_csv(conteudo, csv_nome, marca)
                    if reg:
                        taxa = (reg['cliques'] / reg['entregues'] * 100) if reg['entregues'] > 0 else 0
                        print(f'  ✓ Modelo: {reg["nome"]}')
                        print(f'    Enviadas: {reg["enviadas"]:,} | Entregues: {reg["entregues"]:,} | Cliques: {reg["cliques"]:,} | Conversão: {taxa:.1f}%')
                        todos_registros.append(reg)
        except Exception as e:
            print(f'  ❌ Erro ao processar {zip_file}: {e}')
        print()

    if todos_registros:
        print('=' * 60)
        print(f'  Enviando {len(todos_registros)} registro(s) para o Google Sheets...')
        print('=' * 60)
        inseridos, atualizados = enviar_para_sheets(todos_registros)
        print(f'  ✅ Concluído! {inseridos} inserido(s), {atualizados} atualizado(s).')
        print()
        print('  🌐 Acesse o dashboard para ver os dados:')
        print('  https://victoralmeidacarrera-stack.github.io/dashboard')
        print('=' * 60)
    else:
        print('⚠️  Nenhum registro foi extraído dos arquivos ZIP.')

---

## ℹ️ Informações importantes

**Nomenclatura dos arquivos:**
- O nome do **ZIP** = nome da marca (ex: `GMSP.zip`)
- O nome do **CSV** dentro do ZIP = nome do modelo de mensagem (ex: `TRACKER.csv`)

**Marcas cadastradas no dashboard:**
`GMSP` · `GMBSB` · `Volkswagen` · `GAC` · `GWM` · `Omoda & Jaecoo` · `Zeekr` · `Bajaj` · `Seminovos` · `Nissan`

**Duplicatas:**
Se você importar o mesmo modelo duas vezes, o sistema **atualiza** os dados em vez de duplicar.

**Sessão do Colab:**
O Google Colab encerra a sessão após algum tempo de inatividade. Se isso acontecer, execute as células novamente do início.